In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv()

True

In [3]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [4]:
client = OpenAI(
    api_key=GOOGLE_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [5]:
def generate_text(content):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a sentiment classifier. "
                "The user may provide multiple sentences. "
                "Classify each sentence as Positive, Negative, or Neutral. "
                "Return one label for each sentence in the same order."
            )
        },

        # Few-shot example 1
        {
            "role": "user",
            "content": "I absolutely love this phone!"
        },
        {
            "role": "assistant",
            "content": "Positive"
        },

        # Few-shot example 2
        {
            "role": "user",
            "content": "The food was horrible and cold."
        },
        {
            "role": "assistant",
            "content": "Negative"
        },

        # Few-shot example 3
        {
            "role": "user",
            "content": "The package arrived yesterday."
        },
        {
            "role": "assistant",
            "content": "Neutral"
        },

        # Actual user input
        {
            "role": "user",
            "content": content
        }
    ]

    response = client.chat.completions.create(
        model="gemini-3.6-flash",
        messages=messages,
        max_completion_tokens=300
    )

    return response.choices[0].message.content

In [6]:
result = generate_text(
    "I am very happy with my new laptop."
)

print(result)

Positive


In [7]:
result = generate_text(
    """
    This restaurant was terrible.
    The food was delicious.
    The waiter was very friendly.
    The prices were too high.
    """
)

print(result)

Negative
Positive
Positive
Negative


Now I want to established in context memory for the model to remember the previous conversation and use it in future interactions.

In [11]:
conversation = []


def add_to_conversation(user_message):
    """Send a message to the model and add the response to the conversation."""

    # Add user's message
    conversation.append({
        "role": "user",
        "content": user_message
    })

    # Send the complete conversation
    response = client.chat.completions.create(
        model="gemini-3.6-flash",
        messages=conversation,
        temperature=0.7,
        max_completion_tokens=300
    )

    # Get assistant's response
    assistant_message = response.choices[0].message.content

    # Add assistant's response to conversation
    conversation.append({
        "role": "assistant",
        "content": assistant_message
    })

    return assistant_message

In [12]:
generated_response = add_to_conversation("Hello! How are you?")
print(generated_response)

Hello! I'm doing well, thank you for asking. How are you doing today? How can I help you?


In [13]:
generated_response = add_to_conversation("My name is ABC. What is your name?")
print(generated_response)

Nice to meet you, ABC! 

My name is Gemini. I'm a large language model created by Google. How can I help you today?


In [14]:
generated_response = add_to_conversation("What is my name?")
print(generated_response)

Your name is ABC!


In [15]:
conversation

[{'role': 'user', 'content': 'Hello! How are you?'},
 {'role': 'assistant',
  'content': "Hello! I'm doing well, thank you for asking. How are you doing today? How can I help you?"},
 {'role': 'user', 'content': 'My name is ABC. What is your name?'},
 {'role': 'assistant',
  'content': "Nice to meet you, ABC! \n\nMy name is Gemini. I'm a large language model created by Google. How can I help you today?"},
 {'role': 'user', 'content': 'What is my name?'},
 {'role': 'assistant', 'content': 'Your name is ABC!'}]